# FER2013 — არქიტექტურა 4: Transfer Learning (MobileNetV2)

ImageNet-ზე წინასწარ გავარჯიშებული MobileNetV2, FER2013-ზე ადაპტირებული.

**ორფაზიანი სწავლება:**
- Phase 1 (10 epoch): backbone გაყინულია, მხოლოდ head სწავლობს
- Phase 2 (30 epoch): ყველაფერი გახსნილია, LR=1e-4 (10x პატარა)

| Run | აღწერა |
|---|---|
| arch4_phase1_head_only | Phase 1 — გაყინული backbone |
| arch4_phase2_fine_tune | Phase 2 — სრული fine-tune |
| arch4_mobilenet_scratch | კონტროლი — pretrained weights-ის გარეშე |

## დაყენება და იმპორტი

In [ ]:
!pip install -q wandb

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report

import wandb

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

## WandB კავშირი

In [ ]:
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret('WANDB_API_KEY'))

## მონაცემების ჩატვირთვა

In [ ]:
CSV_PATH = '/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/icml_face_data.csv'
EMOTION_LABELS = {0:'Angry', 1:'Disgust', 2:'Fear', 3:'Happy', 4:'Sad', 5:'Surprise', 6:'Neutral'}

class FER2013Dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.pixels    = dataframe['pixels'].tolist()
        self.labels    = dataframe['emotion'].tolist()
        self.transform = transform
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        img = Image.fromarray(
            np.array(self.pixels[idx].split(), dtype=np.uint8).reshape(48, 48), mode='L'
        )
        if self.transform: img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.long)

def get_transform(augment=False):
    norm = transforms.Normalize([0.5], [0.5])
    if augment:
        return transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.RandomCrop(48, padding=4),
            transforms.ToTensor(), norm
        ])
    return transforms.Compose([transforms.ToTensor(), norm])

df       = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
train_df = df[df['Usage'] == 'Training'].reset_index(drop=True)
val_df   = df[df['Usage'] == 'PublicTest'].reset_index(drop=True)
test_df  = df[df['Usage'] == 'PrivateTest'].reset_index(drop=True)

counts        = train_df['emotion'].value_counts().sort_index().values
class_weights = torch.tensor(1.0 / counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum() * 7

def make_loaders(batch_size=32, augment=True):
    tr = DataLoader(FER2013Dataset(train_df, get_transform(augment)), batch_size, shuffle=True,  num_workers=2)
    vl = DataLoader(FER2013Dataset(val_df,   get_transform(False)),   batch_size, shuffle=False, num_workers=2)
    return tr, vl

print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')

## მოდელის კონსტრუქცია

პირველი Conv ჩანაცვლდება 3-channel → 1-channel. წინასწარი RGB წონები საშუალოდ ინიციალდება.

In [ ]:
def build_mobilenet(dropout=0.5, pretrained=True, freeze_backbone=False):
    if pretrained:
        backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    else:
        backbone = models.mobilenet_v2(weights=None)

    orig = backbone.features[0][0]
    new_conv = nn.Conv2d(1, orig.out_channels, orig.kernel_size,
                         orig.stride, orig.padding, bias=False)
    if pretrained:
        with torch.no_grad():
            new_conv.weight.data = orig.weight.data.mean(dim=1, keepdim=True)
    backbone.features[0][0] = new_conv

    if freeze_backbone:
        for p in backbone.features.parameters():
            p.requires_grad = False

    backbone.classifier = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(backbone.last_channel, 256),
        nn.ReLU(),
        nn.Dropout(dropout * 0.5),
        nn.Linear(256, 7),
    )
    return backbone


m = build_mobilenet(pretrained=True, freeze_backbone=True)
total     = sum(p.numel() for p in m.parameters())
trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'სულ პარამეტრები:      {total:,}')
print(f'სასწავლო პარამეტრები: {trainable:,} ({100*trainable/total:.1f}%)')

## Forward და Backward Pass შემოწმება

In [ ]:
m = build_mobilenet(pretrained=True, freeze_backbone=True).to(DEVICE)
m.eval()
dummy = torch.randn(4, 1, 48, 48).to(DEVICE)
with torch.no_grad():
    out = m(dummy)
assert out.shape == (4, 7)
assert not torch.isnan(out).any()
print(f'Forward pass OK — output shape: {out.shape}')

In [ ]:
m.train()
m.zero_grad()
dummy_labels = torch.randint(0, 7, (4,)).to(DEVICE)
nn.CrossEntropyLoss()(m(dummy), dummy_labels).backward()

print('Phase 1 — backbone გაყინულია, მხოლოდ head-ს აქვს gradient:')
for name, p in m.named_parameters():
    if 'classifier' in name and p.grad is not None:
        print(f'  {name}: {p.grad.abs().mean():.2e}')

m.zero_grad()
print('Backward pass OK')

## სწავლების ციკლი

In [ ]:
def compute_grad_norm(model):
    return sum(p.grad.detach().norm(2).item()**2
               for p in model.parameters() if p.grad is not None) ** 0.5

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    tl, correct, total, gn = 0.0, 0, 0, 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        gn = compute_grad_norm(model)
        optimizer.step()
        tl      += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total   += imgs.size(0)
    return tl / total, correct / total, gn

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    tl, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out  = model(imgs)
        loss = criterion(out, labels)
        tl      += loss.item() * imgs.size(0)
        preds    = out.argmax(1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return tl / total, correct / total, all_preds, all_labels


def run_experiment(run_name, config, model):
    run = wandb.init(
        project='fer2013', name=run_name, config=config,
        tags=['architecture_4', 'transfer_learning'], reinit=True
    )

    t_ldr, v_ldr = make_loaders(batch_size=config['batch_size'], augment=True)
    w         = class_weights.to(DEVICE) if config.get('use_class_weights') else None
    criterion = nn.CrossEntropyLoss(weight=w)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=config['lr'], weight_decay=config.get('weight_decay', 1e-4)
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])

    best = 0.0
    for epoch in range(1, config['epochs'] + 1):
        tr_l, tr_a, gn         = train_epoch(model, t_ldr, criterion, optimizer)
        vl_l, vl_a, preds, lbs = evaluate(model, v_ldr, criterion)
        scheduler.step()

        wandb.log({
            'epoch': epoch,
            'train/loss': tr_l, 'train/accuracy': tr_a,
            'val/loss':   vl_l, 'val/accuracy':   vl_a,
            'grad_norm':  gn,   'learning_rate':  optimizer.param_groups[0]['lr'],
        }, step=epoch)

        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoch {epoch:2d} | train={tr_a:.3f} val={vl_a:.3f} gap={tr_a-vl_a:+.3f} | gn={gn:.3f}')

        if vl_a > best:
            best = vl_a
            torch.save(model.state_dict(), f'{run_name}_best.pt')

    cm = confusion_matrix(lbs, preds, labels=list(range(7)))
    wandb.log({'confusion_matrix': wandb.Table(
        columns=['True \\ Pred'] + list(EMOTION_LABELS.values()),
        data=[[EMOTION_LABELS[i]] + r.tolist() for i, r in enumerate(cm)]
    )})
    wandb.summary['best_val_accuracy'] = best
    run.finish()
    print(f'Best val acc: {best:.4f}\n')
    return best

## ექსპერიმენტი A — Phase 1: გაყინული Backbone

In [ ]:
model_A = build_mobilenet(dropout=0.5, pretrained=True, freeze_backbone=True).to(DEVICE)
acc_A   = run_experiment('arch4_phase1_head_only', {
    'architecture': 'mobilenet_v2_pretrained', 'phase': 'head_only',
    'pretrained': True, 'freeze_backbone': True,
    'dropout': 0.5, 'lr': 1e-3, 'weight_decay': 1e-4,
    'batch_size': 32, 'use_class_weights': True, 'epochs': 10
}, model_A)

## ექსპერიმენტი B — Phase 2: სრული Fine-Tuning

Phase 1-ის checkpoint ჩაიტვირთება, backbone გაიხსნება, LR=1e-4.

In [ ]:
model_B = build_mobilenet(dropout=0.5, pretrained=True, freeze_backbone=False).to(DEVICE)
model_B.load_state_dict(torch.load('arch4_phase1_head_only_best.pt', map_location=DEVICE))

trainable = sum(p.numel() for p in model_B.parameters() if p.requires_grad)
print(f'სასწავლო პარამეტრები Phase 2-ში: {trainable:,}')

model_B.train()
model_B.zero_grad()
dummy = torch.randn(4, 1, 48, 48).to(DEVICE)
dummy_lbs = torch.randint(0, 7, (4,)).to(DEVICE)
nn.CrossEntropyLoss()(model_B(dummy), dummy_lbs).backward()

print('Phase 2 backward — პირველი layer-ები:')
for name, p in list(model_B.named_parameters())[:4]:
    status = f'{p.grad.abs().mean():.2e}' if p.grad is not None else 'None'
    print(f'  {name}: {status}')
model_B.zero_grad()
print('OK — backbone-ს gradient მიეწოდება')

In [ ]:
acc_B = run_experiment('arch4_phase2_fine_tune', {
    'architecture': 'mobilenet_v2_pretrained', 'phase': 'fine_tune',
    'pretrained': True, 'freeze_backbone': False,
    'dropout': 0.5, 'lr': 1e-4, 'weight_decay': 1e-4,
    'batch_size': 32, 'use_class_weights': True, 'epochs': 30
}, model_B)

## ექსპერიმენტი C — Scratch (კონტროლი)

იგივე MobileNetV2 არქიტექტურა, pretrained weights-ის გარეშე. pretraining-ის სარგებლის საჩვენებლად.

In [ ]:
model_C = build_mobilenet(dropout=0.5, pretrained=False, freeze_backbone=False).to(DEVICE)
acc_C   = run_experiment('arch4_mobilenet_scratch', {
    'architecture': 'mobilenet_v2_scratch', 'phase': 'full_scratch',
    'pretrained': False, 'freeze_backbone': False,
    'dropout': 0.5, 'lr': 1e-3, 'weight_decay': 1e-4,
    'batch_size': 32, 'use_class_weights': True, 'epochs': 40
}, model_C)

## საბოლოო შედეგები — ყველა არქიტექტურა

In [ ]:
all_results = {
    'Arch1: TinyCNN':            0.45,
    'Arch2: No Dropout':         0.58,
    'Arch2: Full Reg.':          0.62,
    'Arch3: DeepCNN':            0.65,
    'Arch4: Scratch':            acc_C,
    'Arch4: Phase 1':            acc_A,
    'Arch4: Fine-Tuned':         acc_B,
}

for k, v in all_results.items():
    best = ' <- საუკეთესო' if v == max(all_results.values()) else ''
    print(f'{k:<28} {v:.4f}{best}')

colors = ['#d73027','#fc8d59','#fee090','#91cf60','#4575b4','#74add1','#313695']
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(all_results.keys(), all_results.values(), color=colors)
ax.bar_label(bars, fmt='{:.3f}', padding=2)
ax.set_title('ყველა არქიტექტურა — საბოლოო შედარება')
ax.set_ylabel('Best Val Accuracy')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
_, v_ldr = make_loaders(batch_size=32, augment=False)
_, _, preds, labels = evaluate(model_B, v_ldr, nn.CrossEntropyLoss())
print(classification_report(labels, preds, target_names=list(EMOTION_LABELS.values())))

## Kaggle Submission

In [ ]:
test_ds  = FER2013Dataset(test_df, get_transform(augment=False))
test_ldr = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

model_B.eval()
all_preds = []
with torch.no_grad():
    for imgs, _ in test_ldr:
        all_preds.extend(model_B(imgs.to(DEVICE)).argmax(1).cpu().numpy())

sub = pd.DataFrame({'emotion': all_preds})
sub.index.name = 'id'
sub.to_csv('submission.csv')
print(f'submission.csv შენახულია — {len(sub)} პრედიქცია')